# 跨答案符号的迁移预测

固定 ABC 校准系数，比较 ABC、XYZ、123 下的预测与干预结果。复用 96 个验证家族，不重新拟合。

需要已有验证结果或 validation_Output.zip。Qwen-27B 与 Gemma-12B 分别运行；Qwen 建议 96 GB GPU。

[实验设置与运行说明](../../../docs/experiment_notes/02_section43_format_generalization/647373aeee1ea608.md)


In [ ]:
# Run once. Every later code cell reports both cell and inner-loop progress.
import re
import threading
import time
from IPython.core.magic import register_cell_magic
from tqdm.auto import tqdm

_SETUP_PROGRESS = tqdm(total=1, desc='Initialize progress display', unit='cell')
_ACTIVE_CELL_PROGRESS = None

def cell_stage(message):
    if _ACTIVE_CELL_PROGRESS is not None:
        _ACTIVE_CELL_PROGRESS['stage'] = str(message)
        elapsed = time.monotonic() - _ACTIVE_CELL_PROGRESS['started']
        _ACTIVE_CELL_PROGRESS['bar'].set_postfix_str(
            f"{message} | elapsed {elapsed:.0f}s", refresh=True
        )

@register_cell_magic
def cell_progress(line, cell):
    global _ACTIVE_CELL_PROGRESS
    specification = line.strip() or 'Cell progress'
    match = re.fullmatch(r'(\d+)/(\d+)\s+(.+)', specification)
    if match:
        current, total, label = int(match.group(1)), int(match.group(2)), match.group(3)
        initial = current - 1
    else:
        current, total, label, initial = 1, 1, specification, 0
    started = time.monotonic()
    bar = tqdm(
        total=total, initial=initial, desc=label, unit='cell',
        dynamic_ncols=True, leave=True,
    )
    state = {'bar': bar, 'started': started, 'stage': 'starting'}
    _ACTIVE_CELL_PROGRESS = state
    stopped = threading.Event()

    def heartbeat():
        while not stopped.wait(1.0):
            elapsed = time.monotonic() - started
            bar.set_postfix_str(
                f"{state['stage']} | elapsed {elapsed:.0f}s", refresh=True
            )

    worker = threading.Thread(target=heartbeat, daemon=True)
    worker.start()
    succeeded = False
    try:
        cell_stage('running')
        result = get_ipython().run_cell(cell)
        result.raise_error()
        succeeded = True
        return result.result
    finally:
        stopped.set()
        worker.join(timeout=2.0)
        elapsed = time.monotonic() - started
        if succeeded:
            bar.n = current
            bar.set_postfix_str(f"complete | elapsed {elapsed:.0f}s", refresh=True)
        else:
            bar.set_postfix_str(f"failed | elapsed {elapsed:.0f}s", refresh=True)
        bar.close()
        _ACTIVE_CELL_PROGRESS = None

_SETUP_PROGRESS.update(1)
_SETUP_PROGRESS.close()
print('Cell progress helper ready: 17 tracked cells.')

IMPLEMENTATION_SHA256 = '160372af558929c42134517d3ffc09527278d018135dd39a46f32eb6987dde7d'

In [ ]:
%%cell_progress 1/17 Mount Drive and locate project
cell_stage('Mounting Google Drive')
from google.colab import drive
drive.mount('/content/drive')

cell_stage('Locating VisualTorque project')
from pathlib import Path
PROJECT_CANDIDATES = (
    Path('/content/drive/MyDrive/VisualTorque'),
    Path('/content/drive/MyDrive/VisualTorque/VisualTorque'),
)
PROJECT_ROOT = next(
    (
        path for path in PROJECT_CANDIDATES
        if (path / 'toolkit' / 'pyproject.toml').is_file()
    ),
    None,
)
assert PROJECT_ROOT is not None, (
    'Could not find VisualTorque/toolkit/pyproject.toml on Drive.'
)
TOOLKIT_ROOT = PROJECT_ROOT / 'toolkit'
print('Project:', PROJECT_ROOT)


In [ ]:
%%cell_progress 2/17 Select model and prior results
# Run Qwen-27B first; use a fresh GPU runtime for Gemma afterwards.
MODEL_KEY = 'qwen27b'     # qwen27b / gemma12b
RUN_ID = 'main'
SOURCE_RUN_ID = 'main'   # Previous intervention_prediction_v1 run.
SOURCE_VALIDATION_ZIP = ''  # Optional: '/content/VisualTorque_Intervention_Prediction_qwen27b_validation_Output.zip'
SOURCE_ROOT_OVERRIDE = ''   # Optional prior run DIRECTORY containing support.json.
SOURCE_IMAGES_OVERRIDE = '' # Optional prior images folder; originals on Drive are used first.
# No calibration, layer search, dose search, or sample filtering is performed.


In [ ]:
%%cell_progress 3/17 Install inference environment
import importlib
import os

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import subprocess
import sys

cell_stage('Removing unused TorchAudio')
subprocess.run(
    [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchaudio'],
    check=False,
)
cell_stage('Installing pinned inference and analysis dependencies')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'transformers==5.14.0', 'accelerate>=1.10', 'sentencepiece',
    'protobuf', 'huggingface_hub>=0.34', 'hf_xet>=1.1', 'tqdm>=4.66',
    'pandas>=2.2', 'matplotlib>=3.8',
])

cell_stage('Registering the live VisualTorque source tree')
SRC_ROOT = TOOLKIT_ROOT / 'src'
PACKAGE_ROOT = SRC_ROOT / 'visual_torque_toolkit'
assert (PACKAGE_ROOT / '__init__.py').is_file(), PACKAGE_ROOT
source_text = str(SRC_ROOT)
sys.path[:] = [entry for entry in sys.path if entry != source_text]
sys.path.insert(0, source_text)
for module_name in tuple(sys.modules):
    if (
        module_name == 'visual_torque_toolkit'
        or module_name.startswith('visual_torque_toolkit.')
    ):
        del sys.modules[module_name]
importlib.invalidate_caches()

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchvision
import visual_torque_toolkit
from IPython.display import display
from transformers import AutoModelForMultimodalLM, AutoProcessor
from visual_torque_toolkit.behavior.adapter import HuggingFaceMultimodalAdapter

required_features = (
    '_template', '_to_device', '_candidate_plan', '_append_text_prefix',
)
assert all(
    hasattr(HuggingFaceMultimodalAdapter, name)
    for name in required_features
)
toolkit_path = Path(visual_torque_toolkit.__file__).resolve()
assert PACKAGE_ROOT.resolve() in toolkit_path.parents, toolkit_path
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('torchvision:', torchvision.__version__)
print('Toolkit source:', toolkit_path)


In [ ]:
%%cell_progress 4/17 Load frozen-prediction and provenance utilities
import hashlib, json, math
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
def plain(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    raise TypeError(type(value).__name__)

def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=plain, allow_nan=False).encode()).hexdigest()

def file_hash(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2, default=plain, allow_nan=False) + '\n', encoding='utf-8')
    temp.replace(path)

def freeze(path, identity):
    identity = json.loads(json.dumps(identity, default=plain, sort_keys=True, allow_nan=False))
    if Path(path).is_file():
        assert read_json(path) == identity, f'Configuration changed; preserve the old run and use a new RUN_ID: {path}'
    else:
        save_json(path, identity)
    return digest(identity)
from datetime import datetime, timezone
SEMANTICS=('left','balanced','right')
LABELS=('A','B','C')
SCHEMA='prospective-intervention-effects-v1'
def utc_now():
    return datetime.now(timezone.utc).isoformat()

def center(x):
    x = np.asarray(x, dtype=float)
    return x - x.mean(axis=-1, keepdims=True)

def bf16(x):
    """Round finite FP32 values to BF16, ties-to-even; return FP32 numpy values."""
    a = np.asarray(x, dtype=np.float32).copy()
    assert np.isfinite(a).all()
    u = a.view(np.uint32)
    u = (u + np.uint32(0x7fff) + ((u >> 16) & 1)) & np.uint32(0xffff0000)
    return u.view(np.float32)

def state_hash(x):
    return hashlib.sha256(np.asarray(x, dtype='<f4').tobytes()).hexdigest()

def mapping_reference(fid):
    offset = int(digest(['mapping', fid])[:8], 16) % 3
    sem = list(SEMANTICS[offset:] + SEMANTICS[:offset])
    return [dict(position=i, label=LABELS[i], semantic=sem[i]) for i in range(3)]

def mapping_conflict(reference, semantic, reverse=False):
    pos = next(x['position'] for x in reference if x['semantic'] == semantic)
    label = reference[pos]['label']
    ps = [i for i in range(3) if i != pos]
    ss = [s for s in SEMANTICS if s != semantic]
    ls = [s for s in LABELS if s != label]
    if reverse:
        ss.reverse()
        ls.reverse()
    result = [None]*3
    result[pos] = dict(position=pos, label=ls[0], semantic=ss[0])
    result[ps[0]] = dict(position=ps[0], label=label, semantic=ss[1])
    result[ps[1]] = dict(position=ps[1], label=ls[1], semantic=semantic)
    assert len({next(x['label'] for x in result if x['semantic'] == semantic),label,result[pos]['label']}) == 3
    return result

def transport_vector(values, source_mapping, recipient_mapping, coordinate):
    values = np.asarray(values, dtype=float)  # Always A/B/C order.
    source_index = {x[coordinate]: LABELS.index(x['label']) for x in source_mapping}
    target = {x['label']: values[source_index[x[coordinate]]] for x in recipient_mapping}
    return center([target[label] for label in LABELS])

def predictor_features(row):
    """Explicit allowlist. No recipient gold, CF scores or post-intervention state."""
    s0 = np.asarray(row['s0'], float)
    source_delta = np.asarray(row['source_delta'], float)
    sm, rm = row['source_mapping'], row['recipient_mapping']
    features = {}
    for name, coord in [('semantic','semantic'),('letter','label'),('position','position')]:
        features['source_'+name] = transport_vector(source_delta, sm, rm, coord).tolist()
        # Source argmax endpoint changes; unchanging source decisions give zero.
        cat = np.eye(3)[row['source_after_argmax']] - np.eye(3)[row['source_before_argmax']]
        features['choice_'+name] = transport_vector(cat, sm, rm, coord).tolist()
        features['copy_'+name] = transport_vector(np.eye(3)[row['source_after_argmax']],sm,rm,coord).tolist()
    weights = np.exp(s0-s0.max()); probs = weights/weights.sum()
    gap = float(np.sort(s0)[-1]-np.sort(s0)[-2])
    conf = [*s0, gap, float(-np.sum(probs*np.log(probs+1e-12))),
            float(row['planned_norm']), float(row['relative_norm'])]
    features['confidence'] = conf
    if row['module'] == 'physical':
        axis = np.array([{'left':1., 'balanced':0., 'right':-1.}[
            next(x['semantic'] for x in rm if x['label'] == label)] for label in LABELS])
        sign = 1 if row['target_side'] == 'left' else -1
        signed_dw = sign*int(row['dw'])
        features['recipient_weight'] = (axis*signed_dw*row['recipient_distance']).tolist()
        features['source_torque'] = (axis*signed_dw*row['source_distance']).tolist()
        # Privileged structured-input baseline, explicitly labelled in the protocol.
        features['physical_difficulty'] = conf + list(map(float, row['baseline_physics']))
    return features

def ridge_inputs(row, name):
    f = predictor_features(row)
    if name in ('confidence', 'physical_difficulty'):
        return f[name]
    # Same answer/confidence baseline on both sides of the nested comparison.
    values = f['physical_difficulty'] + f['source_semantic'] + f['source_letter'] + f['source_position']
    if name == 'behavior_plus_weight':
        return values + f['recipient_weight']
    if name == 'behavior_plus_source_torque':
        return values + f['source_torque']
    if name == 'behavior':
        return values
    raise KeyError(name)

def predict_one(row, fitted):
    if fitted['kind'] == 'null':
        return np.zeros(3)
    if fitted['kind'] == 'scalar':
        value = row['natural_input_delta'] if fitted['name']=='natural_response' else predictor_features(row)[fitted['name']]
        return center(np.asarray(value)*fitted['beta'])
    x = np.asarray(ridge_inputs(row, fitted['name']))
    z = np.r_[1., (x-np.asarray(fitted['mean']))/np.asarray(fitted['scale'])]
    return center(z@np.asarray(fitted['coef']))

def operation_key(row):
    return f"{row['module']}:L{row['layer']}"

def seal_predictions(path, rows, fitted, identity):
    """Write once and verify on resume. Call before the first validation patch."""
    assert len({r['trial_id'] for r in rows}) == len(rows)
    records = []
    for r in rows:
        models = fitted[operation_key(r)]['models']
        records.append(dict(trial_id=r['trial_id'], family_id=r['family_id'],
                            input_sha256=digest({k:v for k,v in r.items() if k not in ('gold','cf_gold')}),
                            predicted_deltas={n:predict_one(r,m).tolist() for n,m in models.items()}))
    body = dict(identity=identity, records=records)
    if Path(path).exists():
        old = read_json(path)
        assert old['payload'] == body, 'Predictions or their inputs changed after sealing.'
    else:
        save_json(path, dict(created_at_utc=utc_now(), payload=body))
    return file_hash(path)

def paired_interval(values, key, replicates=10000):
    """Family bootstrap CI and two-sided paired sign-flip test."""
    x = np.asarray(values, float)
    assert len(x) and np.isfinite(x).all()
    rng = np.random.default_rng(int(digest(key)[:16],16))
    draws, null = [], []
    for start in range(0,replicates,250):
        n = min(250,replicates-start)
        draws.extend(rng.choice(x,(n,len(x)),replace=True).mean(1))
        null.extend((rng.choice([-1.,1.],(n,len(x)))*x).mean(1))
    lo, hi = np.quantile(draws,[.025,.975])
    p = (1+int(np.sum(np.abs(null) >= abs(x.mean())-1e-15)))/(replicates+1)
    return dict(n_families=len(x), mean=float(x.mean()), lower=float(lo), upper=float(hi), p_value=float(p))

def holm(values):
    p = np.asarray(values,float); order = np.argsort(p); q = np.empty(len(p)); previous = 0.
    for rank,i in enumerate(order):
        previous = max(previous,min(1.,float(p[i])*(len(p)-rank)))
        q[i] = previous
    return q

def transition(before, after, gold):
    if before != gold and after == gold: return 'repair'
    if before == gold and after != gold: return 'harm'
    if before != after: return 'wrong_to_other_wrong'
    return 'unchanged'
"""Data/provenance and analysis for frozen-predictor format transfer (no fitting)."""
import io
import zipfile
from concurrent.futures import ThreadPoolExecutor

FORMAT_SYMBOLS = {'ABC': ('A', 'B', 'C'), 'XYZ': ('X', 'Y', 'Z'), 'DIGITS': ('1', '2', '3')}
TRANSFER_SCHEMA = 'frozen-format-transfer-v1'


class PriorResults:
    """Read a previous validation ZIP or original run directory without modifying it."""
    def __init__(self, path, model_key):
        self.path = Path(path)
        self.model_key = model_key
        self.prefix = f'models/{model_key}/'
        self.is_zip = self.path.is_file() and self.path.suffix.lower() == '.zip'
        self.archive_prefix = ''
        if self.is_zip:
            with zipfile.ZipFile(self.path) as z:
                candidates = [n[:-len('support.json')] for n in z.namelist()
                              if n == 'support.json' or n.endswith('/support.json')]
                assert len(candidates) == 1, 'ZIP must contain exactly one prior support.json.'
                self.archive_prefix = candidates[0]
        else:
            assert (self.path/'support.json').is_file(), 'SOURCE_ROOT must be the original run root containing support.json.'
        self.manifest = self.json('export_manifest.json') if self.exists('export_manifest.json') else None
        self.hashes = {}

    def exists(self, name):
        if self.is_zip:
            with zipfile.ZipFile(self.path) as z:
                return self.archive_prefix+name in z.namelist()
        return (self.path/name).is_file()

    def raw(self, name):
        assert not Path(name).is_absolute() and '..' not in Path(name).parts
        if self.is_zip:
            with zipfile.ZipFile(self.path) as z:
                data = z.read(self.archive_prefix+name)
        else:
            data = (self.path/name).read_bytes()
        sha = hashlib.sha256(data).hexdigest()
        if getattr(self, 'manifest', None) is not None and name != 'export_manifest.json':
            assert self.manifest['files'].get(name) == sha, 'Prior ZIP member checksum mismatch: '+name
        if hasattr(self, 'hashes'):
            self.hashes[name] = sha
        return data

    def json(self, name):
        return json.loads(self.raw(name))

    def validate(self):
        self.support = self.json('support.json')
        self.protocol = self.json('protocol.json')
        self.runtime = self.json(self.prefix+'runtime_identity.json')
        self.frozen = self.json(self.prefix+'frozen_predictors.json')
        self.completion = self.json(self.prefix+'validation/completion.json')
        self.pre_manifest = self.json(self.prefix+'validation/pre_intervention_manifest.json')
        self.image_manifest = self.json('image_manifest.json')
        self.old_rows = self.json(self.prefix+'validation/target_rows.json')
        self.old_sealed = self.json(self.prefix+'validation/predictions_before_intervention.json')
        self.old_seal = self.json(self.prefix+'validation/prediction_seal.json')
        ident = self.frozen['identity']
        assert ident['model_key'] == self.model_key
        for field, name in [('support_sha256','support.json'),
                            ('runtime_sha256',self.prefix+'runtime_identity.json')]:
            assert ident[field] == self.hashes[name], 'Prior identity mismatch: '+field
        # The original protocol identity is a canonical-object digest; the two
        # other identities above are byte hashes. Do not confuse the contracts.
        assert ident['protocol_sha256'] == digest(self.protocol)
        assert self.support['protocol_sha256'] == digest(self.protocol)
        assert self.completion['status'] == 'heldout_validation_complete'
        assert self.completion['prediction_sha256'] == self.hashes[self.prefix+'validation/predictions_before_intervention.json']
        for key, name in [('prediction_sha256','validation/predictions_before_intervention.json'),
                          ('pre_manifest_sha256','validation/pre_intervention_manifest.json'),
                          ('frozen_models_sha256','frozen_predictors.json')]:
            assert self.old_seal[key] == self.hashes[self.prefix+name]
        self.families = [f for f in self.support['families'] if f['module']=='content' and f['split']=='validation']
        assert len(self.families) == 96 and len({f['family_id'] for f in self.families}) == 96
        calibration_ids = {f['family_id'] for f in self.support['families'] if f['split']=='calibration'}
        assert not calibration_ids & {f['family_id'] for f in self.families}
        self.old_rows = [r for r in self.old_rows if r['module']=='content']
        assert len(self.old_rows)==1152 and len({r['trial_id'] for r in self.old_rows})==1152
        expected_ids = {f['family_id'] for f in self.families}
        assert {r['family_id'] for r in self.old_rows} == expected_ids
        assert all(r['split']=='validation' for r in self.old_rows)
        self.operations = {k:v for k,v in self.frozen['operations'].items() if k.startswith('content:')}
        assert len(self.operations)==2
        assert all(not expected_ids & set(v['calibration_families']) for v in self.operations.values())
        return self

    def cache_archives(self, directory):
        """Only the 96 used families: no recursive history scan and no extraction of unrelated files."""
        directory = Path(directory); directory.mkdir(parents=True, exist_ok=True)
        def one(family):
            fid = family['family_id']; name = self.prefix+'validation/pre_intervention/'+fid+'.json'
            raw = self.raw(name); obj = json.loads(raw)
            assert hashlib.sha256(raw).hexdigest() == self.pre_manifest[fid+'.json']
            assert obj['family']==family
            archive = directory/(fid+'.npz')
            if not archive.is_file() or file_hash(archive)!=obj['arrays_sha256']:
                data = self.raw(name[:-5]+'.npz')
                assert hashlib.sha256(data).hexdigest()==obj['arrays_sha256']
                tmp = archive.with_suffix('.tmp'); tmp.write_bytes(data); tmp.replace(archive)
            self.hashes[name[:-5]+'.npz'] = obj['arrays_sha256']
            (directory/(fid+'.json')).write_bytes(raw)
            return fid
        with ThreadPoolExecutor(max_workers=4) as pool:
            list(tqdm(pool.map(one,self.families),total=len(self.families),desc='Read 96 saved families'))
        return directory


def canonical_mapping(mapping, codebook):
    symbols = FORMAT_SYMBOLS[codebook]
    return [dict(x, label=symbols[LABELS.index(x['label'])]) for x in mapping]


def prediction_metric_rows(rows, sealed):
    pred = {r['trial_id']:r for r in sealed['payload']['records']}
    assert len(pred)==len(rows) and set(pred)=={r['trial_id'] for r in rows}
    metrics=[]
    for row in rows:
        entry=pred[row['trial_id']]
        assert entry['input_sha256']==digest({k:v for k,v in row.items() if k not in ('gold','cf_gold','actual_delta','post_record')})
        s0=np.asarray(row['s0']); actual=np.asarray(row['actual_delta'])
        before=int(np.argmax(s0)); after=int(np.argmax(s0+actual)); gold=LABELS.index(row['gold'])
        for name, delta in entry['predicted_deltas'].items():
            delta=np.asarray(delta); predicted=int(np.argmax(s0+delta))
            metrics.append(dict(family_id=row['family_id'],trial_id=row['trial_id'],
                base_trial_id=row['base_trial_id'],codebook=row['codebook'],operation=operation_key(row),
                predictor=name,mapping=row['mapping_name'],task=row['task'],
                mse=float(np.mean((delta-actual)**2)),post_answer_match=int(predicted==after),
                actual_change=int(after!=before),predicted_change=int(predicted!=before),
                baseline_correct=int(before==gold),post_correct=int(after==gold),predicted_correct=int(predicted==gold),
                actual_transition=transition(before,after,gold),predicted_transition=transition(before,predicted,gold)))
    return pd.DataFrame(metrics)


def analyze_transfer(rows, sealed, frozen, outdir, model_key, replicates=10000):
    outdir=Path(outdir); outdir.mkdir(parents=True,exist_ok=True)
    df=prediction_metric_rows(rows,sealed)
    df.to_csv(outdir/'trial_prediction_metrics.csv',index=False)
    keys=['codebook','operation','predictor']
    fam=df.groupby(keys+['family_id'])[['mse','post_answer_match','actual_change','baseline_correct','post_correct','predicted_correct']].mean().reset_index()
    fam.to_csv(outdir/'family_prediction_metrics.csv',index=False)
    summary=[]
    for values,g in df.groupby(keys):
        cb,op,predictor=values; actual=g.actual_change.eq(1)
        fg=fam[(fam.codebook==cb)&(fam.operation==op)&(fam.predictor==predictor)]
        err=paired_interval(fg.mse,[model_key,values,'mse'],replicates)
        summary.append(dict(zip(keys,values),model=model_key,n_families=g.family_id.nunique(),n_trials=len(g),
            mse=err['mean'],mse_lower=err['lower'],mse_upper=err['upper'],
            correct_choices=int(g.post_answer_match.sum()),choice_accuracy=float(g.post_answer_match.mean()),
            actual_changed=int(actual.sum()),correct_when_changed=int(g.loc[actual,'post_answer_match'].sum()),
            accuracy_when_changed=float(g.loc[actual,'post_answer_match'].mean()) if actual.any() else None,
            actual_gain=float((g.post_correct-g.baseline_correct).mean()),
            predicted_gain=float((g.predicted_correct-g.baseline_correct).mean()),
            frozen_selected=predictor==frozen[op]['selected_baseline']))
    summary=pd.DataFrame(summary); summary.to_csv(outdir/'prediction_summary.csv',index=False)
    tests=[]
    # Four comparisons at each of two sites and two NEW codebooks = 16/model.
    for (cb,op),g in fam[fam.codebook.ne('ABC')].groupby(['codebook','operation']):
        pivot=g.pivot(index='family_id',columns='predictor',values='mse')
        selected=frozen[op]['selected_baseline']
        comparisons=[('source_semantic','source_letter'),('source_letter','source_position'),
                     (selected,'null'),(selected,'confidence')]
        for index,(a,b) in enumerate(comparisons):
            stats=paired_interval((pivot[b]-pivot[a]).values,[model_key,cb,op,index,a,b],replicates)
            tests.append(dict(model=model_key,codebook=cb,operation=op,better_if_positive=a,comparator=b,**stats))
    for t,q in zip(tests,holm([t['p_value'] for t in tests])): t['holm_p_within_model']=float(q)
    pd.DataFrame(tests).to_csv(outdir/'primary_comparisons.csv',index=False)
    transfer=[]
    for (op,predictor),g in fam.groupby(['operation','predictor']):
        pivot=g.pivot(index='family_id',columns='codebook',values='mse')
        assert set(pivot)==set(FORMAT_SYMBOLS) and not pivot.isna().any().any()
        for cb in ('XYZ','DIGITS'):
            stats=paired_interval((pivot[cb]-pivot.ABC).values,[model_key,op,predictor,cb,'degradation'],replicates)
            transfer.append(dict(model=model_key,operation=op,predictor=predictor,codebook=cb,
                                 positive_means='larger_error_than_fresh_ABC',**stats))
    pd.DataFrame(transfer).to_csv(outdir/'format_error_change_descriptive.csv',index=False)
    # This is not an equivalence test: a CI containing zero does NOT prove invariance.
    events=[]
    for values,g in df.groupby(keys):
        for event in ('repair','harm','wrong_to_other_wrong','unchanged'):
            actual=g.actual_transition.eq(event); predicted=g.predicted_transition.eq(event)
            nt=int(actual.sum()); npred=int(predicted.sum()); tp=int((actual&predicted).sum())
            events.append(dict(zip(keys,values),event=event,n_actual=nt,n_predicted=npred,n_true_positive=tp,
                recall=tp/nt if nt else None,precision=tp/npred if npred else None))
    pd.DataFrame(events).to_csv(outdir/'transition_events.csv',index=False)
    mapping=df.groupby(keys+['mapping','family_id'])[['baseline_correct','post_correct','predicted_correct']].mean().reset_index()
    mapping['actual_gain']=mapping.post_correct-mapping.baseline_correct
    mapping['predicted_gain']=mapping.predicted_correct-mapping.baseline_correct
    mapping.to_csv(outdir/'mapping_family_gains.csv',index=False)
    return summary,pd.DataFrame(tests)

IMPLEMENTATION_SHA256='9c759116cd001e094555d7bc8da0292705f9c64db6f9d8625f65defbd680083e'


In [ ]:
%%cell_progress 5/17 Read prior support and frozen predictors
assert MODEL_KEY in ('qwen27b','gemma12b')
assert RUN_ID and all(c.isalnum() or c in '_-' for c in RUN_ID)
assert SOURCE_RUN_ID and all(c.isalnum() or c in '_-' for c in SOURCE_RUN_ID)
prior_root=Path(SOURCE_ROOT_OVERRIDE) if SOURCE_ROOT_OVERRIDE else PROJECT_ROOT/'results/intervention_prediction_v1'/SOURCE_RUN_ID
zip_name=f'VisualTorque_Intervention_Prediction_{MODEL_KEY}_validation_Output.zip'
if SOURCE_VALIDATION_ZIP:
    prior_path=Path(SOURCE_VALIDATION_ZIP)
else:
    prior_path=next((p for p in (Path('/content')/zip_name,prior_root/zip_name) if p.is_file()),prior_root)
print('Prior results:',prior_path)
PRIOR=PriorResults(prior_path,MODEL_KEY).validate()
MODEL_SPECS={
 'qwen27b':dict(model_id='Qwen/Qwen3.5-27B',revision='373612cb4f9288fcf550b526d86d85febedd9801',expected_layers=64,content_layers=(49,63)),
 'gemma12b':dict(model_id='google/gemma-4-12B-it',revision='707f0a3b8a3c7ad586ed01e27eafbad8a27dd0f7',expected_layers=48,content_layers=(37,47)),
}
SPEC=MODEL_SPECS[MODEL_KEY]
for key,value in SPEC.items():
    old=PRIOR.protocol['model_specs'][MODEL_KEY][key]
    assert list(value)==old if isinstance(value,tuple) else value==old, 'Prior model/site specification changed.'
MODEL_ID=SPEC['model_id']; MODEL_REVISION=SPEC['revision']; EXPECTED_LAYER_COUNT=SPEC['expected_layers']
MODEL_DTYPE='bfloat16'; DEVICE_MAP='auto'; CAPTURE_LAYERS=SPEC['content_layers']; SEED=2026091601
RUN_ROOT=PROJECT_ROOT/'results/frozen_format_transfer_v1'/RUN_ID
MODEL_ROOT=RUN_ROOT/'models'/MODEL_KEY; PHASE_ROOT=MODEL_ROOT/'validation'
ANALYSIS_DIR=PHASE_ROOT/'analysis'; ANALYSIS_DIR.mkdir(parents=True,exist_ok=True)
IMAGE_DIR=RUN_ROOT/'images'; FAMILIES=PRIOR.families; FITTED=PRIOR.operations
PROTOCOL=dict(schema=TRANSFER_SCHEMA,model_specs=MODEL_SPECS,codebooks=FORMAT_SYMBOLS,
    seed=SEED,support_sha256=PRIOR.frozen['identity']['support_sha256'],n_families=96,
    source_module='content',site='decoder_block_output_last_scored_prompt_token',dose=1.,
    delta='native BF16 source_after-source_before; native BF16 recipient addition',
    implementation=IMPLEMENTATION_SHA256,
    calibration='none; every coefficient and selected baseline copied from previous calibration',
    coordinate_adapter='A/B/C canonical array slots correspond to X/Y/Z or 1/2/3, independent of option position',
    source_recipient_codebook='same codebook on both sides; recipient semantic/letter/position mappings remain crossed',
    scope='previously evaluated geometry families, NEW codebook conditions; no new independent geometries',
    abc='fresh ABC anchor in current runtime; historical ABC is an additional replay comparison',
    digit_instruction='letter replaced by number; all other task/rule wording is unchanged',
    primary='centered candidate score delta MSE; 4 contrasts x 2 sites x 2 new codebooks per model',
    multiple_comparisons='16 per model; 32 across both planned models',
    secondary='changed-only exact choices; repair/harm; paired format error changes are descriptive, not equivalence tests',
    random_controls='not repeated: no new physical-direction specificity claim',
    tail='deterministic norm/head reconstruction; report numerical errors, do not count as learned prediction success')
freeze(RUN_ROOT/'protocol.json',PROTOCOL)
freeze(RUN_ROOT/'support.json',PRIOR.support)
prior_cache_key=digest([MODEL_KEY,PRIOR.frozen['identity']])[:16]
PRIOR_CACHE=PRIOR.cache_archives(Path('/content/visualtorque_format_cache')/prior_cache_key)
SOURCE_IDENTITY=dict(model_key=MODEL_KEY,prior_files_sha256=dict(PRIOR.hashes),
    frozen_predictors_sha256=PRIOR.hashes[PRIOR.prefix+'frozen_predictors.json'],
    verification='consumed ZIP members checked against export manifest' if PRIOR.manifest else 'direct run files frozen; embedded prior identities and activation archive hashes verified')
freeze(MODEL_ROOT/'source_identity.json',SOURCE_IDENTITY)
frozen_path=MODEL_ROOT/'frozen_predictors.json'
frozen_bytes=PRIOR.raw(PRIOR.prefix+'frozen_predictors.json')
if frozen_path.exists(): assert frozen_path.read_bytes()==frozen_bytes
else: frozen_path.write_bytes(frozen_bytes)
freeze(MODEL_ROOT/'archived_content_rows.json',PRIOR.old_rows)
freeze(MODEL_ROOT/'archived_runtime_identity.json',PRIOR.runtime)
freeze(MODEL_ROOT/'archived_protocol.json',PRIOR.protocol)
freeze(MODEL_ROOT/'archived_prediction_seal.json',PRIOR.old_seal)
freeze(MODEL_ROOT/'archived_predictions.json',PRIOR.old_sealed)
OLD_ROWS_BY_FAMILY=defaultdict(list)
for row in PRIOR.old_rows: OLD_ROWS_BY_FAMILY[row['family_id']].append(row)
print('Reusing 96 validation families and frozen predictors. No history scan and no fitting.')
display(pd.DataFrame([dict(operation=k,frozen_selected=v['selected_baseline']) for k,v in FITTED.items()]))


In [ ]:
%%cell_progress 6/17 Load execution definitions
def output_hidden_tensor(output):
    if torch.is_tensor(output):
        hidden = output
    elif isinstance(output, (tuple, list)) and output and torch.is_tensor(output[0]):
        hidden = output[0]
    elif hasattr(output, 'last_hidden_state') and torch.is_tensor(output.last_hidden_state):
        hidden = output.last_hidden_state
    else:
        raise TypeError(f'Unsupported output: {type(output).__name__}')
    assert hidden.ndim == 3
    return hidden

def replace_output_hidden(output, modified):
    if torch.is_tensor(output):
        return modified
    if isinstance(output, tuple):
        return (modified,) + output[1:]
    if isinstance(output, list):
        return [modified] + output[1:]
    raise TypeError(type(output).__name__)

def prepare_examples(examples):
    examples = tuple(examples)
    plan = adapter._candidate_plan(examples[0])
    if plan is None:
        raise RuntimeError('All answer symbols must have single-token continuations after a common prefix.')
    for example in examples[1:]:
        assert adapter._candidate_plan(example) == plan
    fixed_prefix, token_by_label = plan
    inputs = adapter._append_text_prefix(
        adapter._to_device(adapter._template(
            [example.messages() for example in examples],
            add_generation_prompt=True,
        )), fixed_prefix,
    )
    return {'examples': examples, 'inputs': inputs, 'token_by_label': token_by_label}

def final_mask(prepared):
    assert len(prepared['examples']) == 1
    attention = prepared['inputs']['attention_mask'][0].detach().cpu().bool()
    mask = torch.zeros_like(attention)
    mask[int(torch.where(attention)[0][-1])] = True
    return mask

def score_records(prepared, logits):
    logprobs = torch.log_softmax(logits[:, -1, :].float(), dim=-1)
    records = []
    for index, example in enumerate(prepared['examples']):
        scores = [
            float(logprobs[index, prepared['token_by_label'][label]].detach().cpu())
            for label in example.labels
        ]
        maximum = max(scores)
        weights = [math.exp(value - maximum) for value in scores]
        probabilities = [value / sum(weights) for value in weights]
        predicted = int(np.argmax(scores))
        records.append({
            'candidate_labels': list(example.labels),
            'candidate_semantics': list(example.option_semantics),
            'candidate_logprobs': scores,
            'candidate_probabilities': probabilities,
            'predicted_label': example.labels[predicted],
            'predicted_semantic': example.option_semantics[predicted],
            'gold_label': example.gold_label,
            'gold_semantic': example.gold_semantic,
        })
    return records

def example_stamp(example):
    return dict(example_id=example.example_id,prompt=example.prompt,image=file_hash(example.image_path),
        labels=list(example.labels),semantics=list(example.option_semantics),gold=str(example.gold_semantic),
        torque=int(example.net_torque),split=example.split)

def current_runtime_identity():
    import transformers
    return dict(torch=torch.__version__,transformers=transformers.__version__,cuda=torch.version.cuda,
        dtype=MODEL_DTYPE,batch_size=1,checkpoint=MODEL_REVISION,layer_path=LAYER_PATH,
        implementation=IMPLEMENTATION_SHA256,
        toolkit={str(p.relative_to(TOOLKIT_ROOT)):file_hash(p) for p in sorted((TOOLKIT_ROOT/'src').rglob('*.py'))})
"""GPU execution, pre-intervention sealing and exports for the prospective study."""
import shutil
import zipfile
from visual_torque_toolkit.behavior import PromptCondition
from visual_torque_toolkit.behavior.data import BehaviorExample
from visual_torque_toolkit.render import RenderStyle, render_scene
from visual_torque_toolkit.world import Scene, Side, Stack

RULE_TEXT = ('Each square is one unit of weight. Distance is measured in tick '
             'intervals from the center pivot. Torque equals weight times distance. '
             'Total torque on a side is the sum over stacks on that side.')
QUESTION_TEXT = {'outcome':'Which way will the balance scale tip?',
                 'weight_more':'Which side has more total weight?',
                 'weight_less':'Which side has less total weight?'}
SEMANTIC_TEXT = {'left':'left side','balanced':'equal','right':'right side'}


def question_gold(role, task):
    f = role['features']
    value = f['net_torque'] if task == 'outcome' else f['weight_score']*(1 if task == 'weight_more' else -1)
    return 'left' if value>0 else 'right' if value<0 else 'balanced'

def make_predictive_example(family, role_name, task, mapping):
    role = family['roles'][role_name]
    sem = [next(x['semantic'] for x in mapping if x['label']==label) for label in LABELS]
    options = '\n'.join(f"{x['label']}. {SEMANTIC_TEXT[x['semantic']]}" for x in mapping)
    prompt = '\n\n'.join(['Answer this question using only the supplied balance-scale scene.',
        RULE_TEXT, 'Question: '+QUESTION_TEXT[task], 'Options:\n'+options,
        'Do not explain your reasoning. Reply with exactly one option letter and nothing else. Valid replies are: A, B, C.'])
    gold = question_gold(role,task)
    eid = family['family_id']+'__'+role_name+'__'+task+'__'+digest(mapping)[:12]
    return BehaviorExample(example_id=eid,question_id=eid,scene_id=role['scene_id'],
        condition=PromptCondition.IMAGE_RULE,question_kind=task,variant=SCHEMA,
        split=family['split'],problem_type='controlled_four_stack_family',outcome=gold,
        net_torque=int(role['features']['net_torque']),prompt=prompt,
        image_path=IMAGE_DIR/role['image_filename'],labels=LABELS,
        option_text=tuple(SEMANTIC_TEXT[s] for s in sem),option_semantics=tuple(sem),
        gold_label=LABELS[sem.index(gold)],gold_semantic=gold)

def prepared_fingerprint(prepared):
    tensors = {}
    for key, value in prepared['inputs'].items():
        if torch.is_tensor(value):
            x = value.detach().cpu().contiguous()
            raw = x.view(torch.uint8).numpy().tobytes()
            tensors[key] = dict(shape=list(x.shape),dtype=str(x.dtype),sha256=hashlib.sha256(raw).hexdigest())
        else:
            tensors[key] = str(value)
    return digest(dict(inputs=tensors,candidate_tokens=prepared['token_by_label']))

def predictive_forward(example, layer=None, payload=None, random_key=None, expected_pre=None, expected_input=None):
    prepared = prepare_examples([example])
    fingerprint = prepared_fingerprint(prepared)
    if expected_input is not None:
        assert fingerprint == expected_input, 'Prepared prompt/image tensors changed.'
    mask = final_mask(prepared)
    captured, audits, handles = {}, {}, []
    active = CAPTURE_LAYERS if layer is None else (int(layer),)
    for index in active:
        def hook(_module,_inputs,output,index=index):
            hidden = output_hidden_tensor(output)
            m = mask.to(hidden.device)
            assert hidden.dtype == torch.bfloat16 and int(m.sum()) == 1
            pre = hidden[0,m]
            before = pre.detach().float().cpu().numpy().copy()
            if payload is not None:
                assert index == layer
                # Check the current untreated state against the state used to plan this write.
                assert expected_pre is not None
                max_error = float(np.max(np.abs(before-np.asarray(expected_pre))))
                assert max_error == 0., (
                    f'Untreated-state replay differs ({max_error}). Do not loosen the check; '
                    'inspect runtime/input identity and restart this run consistently.')
                delta = torch.as_tensor(np.asarray(payload).copy(),device=pre.device,dtype=pre.dtype).reshape(pre.shape)
                desired = float(((pre+delta).float()-pre.float()).norm())
                if random_key is None:
                    post = pre+delta
                    audit = dict(matched=True,relative_error=0.,target_norm=desired)
                else:
                    cast = lambda x:torch.as_tensor(np.asarray(x).copy(),dtype=torch.bfloat16).float().numpy()
                    post_np,audit = native_norm_edit(before,random_unit(before.shape,random_key),desired,cast)
                    post = torch.as_tensor(post_np,device=pre.device,dtype=pre.dtype)
                audit.update(actual_norm=float((post.float()-pre.float()).norm()),
                             pre_sha256=state_hash(before), planned_payload_sha256=state_hash(payload))
                audits[str(index)] = audit
                modified = hidden.clone(); modified[0,m] = post
                captured[index] = post.detach().float().cpu().numpy().copy()
                return replace_output_hidden(output,modified)
            captured[index] = before
            return None
        handles.append(DECODER_LAYERS[index].register_forward_hook(hook))
    try:
        with torch.inference_mode():
            out = adapter.model(**prepared['inputs'],use_cache=False)
        record = score_records(prepared,out.logits)[0]
    finally:
        for handle in handles:
            handle.remove()
    record.update(input_sha256=fingerprint,write_audits=audits)
    return record,captured

def trial_specs(family):
    source_mapping = mapping_reference(family['family_id'])
    if family['module'] == 'content':
        routes = [dict(source_before='base',source_after='aligned_source',recipient='base',
                       task=task,tag=task) for task in ('weight_more','weight_less')]
    else:
        routes = [dict(source_before=f'source_{ds}_before',source_after=f'source_{ds}_after',
                       recipient=f'recipient_{dr}_before',cf_role=f'recipient_{dr}_after',
                       task='outcome',tag=f'{ds}_{dr}',source_distance=family[ds],
                       recipient_distance=family[dr])
                  for ds in ('near','far') for dr in ('near','far')]
    for route in routes:
        if family['module'] == 'content':
            focus = question_gold(family['roles'][route['source_after']],route['task'])
        else:
            side_sign = 1 if family['target_side']=='left' else -1
            focus = 'left' if family['dw']*side_sign>0 else 'right'
        mappings = [('identity',source_mapping),
                    ('conflict_forward',mapping_conflict(source_mapping,focus)),
                    ('conflict_reverse',mapping_conflict(source_mapping,focus,True))]
        for mapping_name,mapping in mappings:
            for layer in SPEC[family['module']+'_layers']:
                tid = family['family_id']+f"__{route['tag']}__{mapping_name}__L{layer}"
                yield dict(route,trial_id=tid,layer=layer,source_mapping=source_mapping,
                           recipient_mapping=mapping,mapping_name=mapping_name)

def render_support(families):
    IMAGE_DIR.mkdir(parents=True,exist_ok=True)
    for f in tqdm(families,desc='Render frozen support'):
        for r in f['roles'].values():
            path = IMAGE_DIR/r['image_filename']
            if path.exists():
                continue
            scene = Scene(r['scene_id'],tuple(Stack(str(i),Side(s['side']),int(s['distance']),int(s['units']))
                                              for i,s in enumerate(r['stacks'])))
            render_scene(scene,path,RenderStyle(ticks_per_side=6))
"""GPU execution for a fixed codebook transfer protocol; never fits a predictor."""
from dataclasses import replace
import inspect
import shutil

_original_example = make_predictive_example
_original_trial_specs = trial_specs


def prepare_transfer_images():
    """Prefer original PNGs: identical pixels need not have identical PNG compression across platforms."""
    IMAGE_DIR.mkdir(parents=True,exist_ok=True)
    candidates=[]
    if SOURCE_IMAGES_OVERRIDE: candidates.append(Path(SOURCE_IMAGES_OVERRIDE))
    candidates += [prior_root/'images']
    if PRIOR.is_zip: candidates.append(PRIOR.path.parent/'images')
    provenance=[]
    for family in tqdm(FAMILIES,desc='Restore original scene images'):
        for role in family['roles'].values():
            name=role['image_filename']; target=IMAGE_DIR/name; expected=PRIOR.image_manifest[name]
            if target.exists():
                assert file_hash(target)==expected, 'Existing run image differs: '+str(target)
                provenance.append(dict(name=name,method='verified_existing')); continue
            original=next((root/name for root in candidates if (root/name).is_file() and file_hash(root/name)==expected),None)
            if original:
                shutil.copy2(original,target); method='verified_original_copy'
            else:
                scene=Scene(role['scene_id'],tuple(Stack(str(i),Side(s['side']),int(s['distance']),int(s['units']))
                                                 for i,s in enumerate(role['stacks'])))
                # Verify a temporary candidate before making it part of this run.
                temp=target.with_suffix('.tmp.png'); render_scene(scene,temp,RenderStyle(ticks_per_side=6))
                if file_hash(temp)!=expected:
                    temp.unlink()
                    raise RuntimeError('Rendered PNG bytes differ from the previous run. Keep the original images and set SOURCE_IMAGES_OVERRIDE to the old run images folder; do not loosen the checksum. Different Pillow/zlib versions can change PNG bytes.')
                temp.replace(target); method='verified_rerender'
            provenance.append(dict(name=name,method=method))
    save_json(PHASE_ROOT/'image_loading_provenance.json',provenance)


def make_format_example(family, role, task, mapping, codebook):
    ex=_original_example(family,role,task,mapping)
    if codebook=='ABC':
        return ex
    symbols=FORMAT_SYMBOLS[codebook]
    actual=canonical_mapping(mapping,codebook)
    options='\n'.join(f"{x['label']}. {SEMANTIC_TEXT[x['semantic']]}" for x in actual)
    noun='number' if codebook=='DIGITS' else 'letter'
    prompt='\n\n'.join(['Answer this question using only the supplied balance-scale scene.',RULE_TEXT,
        'Question: '+QUESTION_TEXT[task], 'Options:\n'+options,
        f'Do not explain your reasoning. Reply with exactly one option {noun} and nothing else. Valid replies are: '+', '.join(symbols)+'.'])
    return replace(ex,example_id=ex.example_id+'__'+codebook,question_id=ex.question_id+'__'+codebook,
                   prompt=prompt,labels=symbols,gold_label=symbols[LABELS.index(ex.gold_label)])


def format_trial_specs(family, codebook):
    for spec in _original_trial_specs(family):
        assert family['module']=='content'
        yield dict(spec,base_trial_id=spec['trial_id'],trial_id=spec['trial_id']+'__'+codebook,codebook=codebook)


class TailReadout:
    """Actual final norm and vocabulary head; no decoder blocks and no learned surrogate.

    Transformers Qwen3.5/Gemma4 implementations use norm -> lm_head, with Gemma
    applying config-defined final logit softcapping. Save installed source and
    report numerical replay error; do not silently call it an exact BF16 replay.
    """
    def __init__(self):
        parent=resolve_module_path(adapter.model,LAYER_PATH.rsplit('.',1)[0])
        self.norm=parent.norm
        self.head=adapter.model.get_output_embeddings()
        assert self.head is adapter.model.lm_head and isinstance(self.head,torch.nn.Linear)
        self.config=adapter.model.config.get_text_config()
        self.cap=getattr(self.config,'final_logit_softcapping',None)
        self.device=next(self.norm.parameters()).device
        self.head_device=self.head.weight.device
        self.dtype=self.head.weight.dtype
        assert self.dtype==torch.bfloat16
        self.plans={}
        for cb in FORMAT_SYMBOLS:
            ex=make_format_example(FAMILIES[0],'base','outcome',mapping_reference(FAMILIES[0]['family_id']),cb)
            plan=adapter._candidate_plan(ex)
            assert plan is not None, f'{cb} must have three single-token continuations after one shared prefix.'
            prefix,tokens=plan
            assert set(tokens)==set(FORMAT_SYMBOLS[cb]) and len(set(tokens.values()))==3
            self.plans[cb]=dict(prefix=list(prefix),tokens=tokens)
        # A changed scoring position would be an extra manipulation beyond the codebook.
        assert len({tuple(p['prefix']) for p in self.plans.values()})==1, 'Codebooks have different scoring prefixes; inspect the tokenizer protocol before continuing.'
        self.identity=dict(norm_class=type(self.norm).__name__,head_class=type(self.head).__name__,
            norm_path=LAYER_PATH.rsplit('.',1)[0]+'.norm',final_logit_softcapping=self.cap,
            plans=self.plans,model_type=self.config.model_type,head_batch_size=1,
            interpretation='output-path reconstruction, not a learned forecast or physical-mechanism test')
        source='\n\n'.join(inspect.getsource(type(m).forward) for m in (parent,adapter.model,self.norm))
        freeze(PHASE_ROOT/'tail_readout_identity.json',dict(self.identity,installed_source_sha256=hashlib.sha256(source.encode()).hexdigest()))
        (PHASE_ROOT/'installed_tail_source.txt').write_text(source,encoding='utf-8')
        # Export only the small reproducibility subset, never full model weights.
        arrays={'norm_weight':self.norm.weight.detach().float().cpu().numpy()}
        for cb,plan in self.plans.items():
            ids=[plan['tokens'][s] for s in FORMAT_SYMBOLS[cb]]
            arrays[cb+'_head_rows']=self.head.weight[ids].detach().float().cpu().numpy()
            if self.head.bias is not None: arrays[cb+'_head_bias']=self.head.bias[ids].detach().float().cpu().numpy()
        p=PHASE_ROOT/'tail_parameters.npz'; tmp=p.with_suffix('.tmp.npz'); np.savez_compressed(tmp,**arrays); tmp.replace(p)
        freeze(PHASE_ROOT/'tail_parameters_identity.json',dict(sha256=file_hash(p),norm_eps=float(self.norm.eps)))

    def scores(self, hidden, codebook):
        h=torch.as_tensor(np.asarray(hidden).copy(),dtype=self.dtype,device=self.device).reshape(1,1,-1)
        with torch.inference_mode():
            logits=self.head(self.norm(h).to(self.head_device))
            if self.cap is not None:
                logits=torch.tanh(logits/self.cap)*self.cap
            ids=[self.plans[codebook]['tokens'][s] for s in FORMAT_SYMBOLS[codebook]]
            scores=logits[0,-1,ids].float().cpu().numpy()
        # Centered log probabilities equal centered logits; no full-vocabulary softmax needed.
        return center(scores)


def run_archive_tail_audit():
    folder=PHASE_ROOT/'archive_tail_audit'; folder.mkdir(parents=True,exist_ok=True)
    all_rows=[]; last=EXPECTED_LAYER_COUNT-1
    for family in tqdm(FAMILIES,desc='Replay archived final states through output modules'):
        fid=family['family_id']; pre=read_json(PRIOR_CACHE/(fid+'.json'))
        saved=folder/(fid+'.json')
        identity=digest([RUNTIME_IDENTITY,pre['arrays_sha256'],TAIL.identity,
                         SOURCE_IDENTITY['prior_files_sha256'],IMPLEMENTATION_SHA256])
        if saved.exists():
            obj=read_json(saved); assert obj['identity']==identity
            all_rows.extend(obj['rows']); continue
        records=[]; cache={}
        with np.load(PRIOR_CACHE/(fid+'.npz'),allow_pickle=False) as arrays:
            for row in OLD_ROWS_BY_FAMILY[fid]:
                if row['layer']!=last: continue
                h0=arrays[row['pre_key']]; delta=arrays[row['payload_key']]
                assert state_hash(h0)==row['pre_sha256'] and state_hash(delta)==row['payload_sha256']
                if row['pre_key'] not in cache: cache[row['pre_key']]=TAIL.scores(h0,'ABC')
                before=cache[row['pre_key']]; after=TAIL.scores(bf16(h0+delta),'ABC')
                actual_before=np.asarray(row['s0']); actual_after=actual_before+np.asarray(row['actual_delta'])
                records.append(dict(trial_id=row['trial_id'],family_id=fid,
                    before_max_abs_error=float(np.max(np.abs(before-actual_before))),
                    after_max_abs_error=float(np.max(np.abs(after-actual_after))),
                    delta_mse=float(np.mean(((after-before)-row['actual_delta'])**2)),
                    post_answer_match=int(np.argmax(after)==np.argmax(actual_after)),
                    actual_change=int(np.argmax(actual_before)!=np.argmax(actual_after)),
                    reconstructed_before=before.tolist(),reconstructed_after=after.tolist()))
        save_json(saved,dict(identity=identity,rows=records)); all_rows.extend(records)
    assert len(all_rows)==576
    df=pd.DataFrame(all_rows); df.to_csv(PHASE_ROOT/'analysis/archive_tail_replay.csv',index=False)
    print('Archived tail reconstruction; numerical errors are reported, never auto-corrected:')
    display(df[['before_max_abs_error','after_max_abs_error','delta_mse','post_answer_match']].describe())
    save_json(PHASE_ROOT/'archive_tail_summary.json',dict(n_trials=len(df),
        max_before_error=float(df.before_max_abs_error.max()),max_after_error=float(df.after_max_abs_error.max()),
        mean_delta_mse=float(df.delta_mse.mean()),post_answer_match=float(df.post_answer_match.mean()),
        status='reconstruction_reported_not_a_physical_mechanism_test',
        caveat='Replay uses actual modules but token-only GEMM shapes and potentially a different GPU; compare errors, do not assume bitwise equality.'))


def capture_format_family(family, codebook):
    folder=PHASE_ROOT/'pre_intervention'; folder.mkdir(parents=True,exist_ok=True)
    stem=family['family_id']+'__'+codebook
    path=folder/(stem+'.json'); archive=folder/(stem+'.npz')
    identity=digest([RUNTIME_IDENTITY,family,codebook,IMPLEMENTATION_SHA256])
    if path.exists():
        obj=read_json(path)
        assert obj['identity']==identity and file_hash(archive)==obj['arrays_sha256']
        return obj
    arrays={}; records={}; examples={}; rows=[]; tail_clean=[]
    def capture(role,task,mapping):
        ex=make_format_example(family,role,task,mapping,codebook)
        key=digest(ex.example_id)[:20]
        if key not in records:
            rec,hidden=predictive_forward(ex)
            records[key]=rec
            examples[key]=dict(role=role,task=task,canonical_mapping=mapping,
                actual_mapping=canonical_mapping(mapping,codebook),stamp=example_stamp(ex))
            for layer,h in hidden.items(): arrays[key+f'__L{layer}']=h
            tail=TAIL.scores(hidden[EXPECTED_LAYER_COUNT-1],codebook)
            tail_clean.append(dict(example_key=key,max_abs_error=float(np.max(np.abs(tail-center(rec['candidate_logprobs'])))),
                                   scores=tail.tolist(),actual=center(rec['candidate_logprobs']).tolist()))
        return key
    for spec in format_trial_specs(family,codebook):
        sb=capture(spec['source_before'],spec['task'],spec['source_mapping'])
        sa=capture(spec['source_after'],spec['task'],spec['source_mapping'])
        rb=capture(spec['recipient'],'outcome',spec['recipient_mapping']); layer=spec['layer']
        h0=arrays[rb+f'__L{layer}']; delta=bf16(arrays[sa+f'__L{layer}']-arrays[sb+f'__L{layer}'])
        norm=float(np.linalg.norm(bf16(h0+delta)-h0)); payload_key=spec['trial_id']+'__delta'
        arrays[payload_key]=delta
        gold=_original_example(family,spec['recipient'],'outcome',spec['recipient_mapping']).gold_label
        rows.append(dict(spec,family_id=family['family_id'],module='content',split='validation',stratum=family['stratum'],
            gold=gold,source_key_before=sb,source_key_after=sa,recipient_key=rb,payload_key=payload_key,pre_key=rb+f'__L{layer}',
            payload_sha256=state_hash(delta),pre_sha256=state_hash(h0),source_delta=center(np.asarray(records[sa]['candidate_logprobs'])-records[sb]['candidate_logprobs']).tolist(),
            source_before_argmax=int(np.argmax(records[sb]['candidate_logprobs'])),source_after_argmax=int(np.argmax(records[sa]['candidate_logprobs'])),
            s0=center(records[rb]['candidate_logprobs']).tolist(),planned_norm=norm,relative_norm=norm/max(float(np.linalg.norm(h0)),1e-12)))
    temp=archive.with_suffix('.tmp.npz'); np.savez_compressed(temp,**arrays); temp.replace(archive)
    obj=dict(identity=identity,arrays_sha256=file_hash(archive),family_id=family['family_id'],codebook=codebook,
             examples=examples,baseline_records=records,rows=rows,tail_clean=tail_clean,created_at_utc=utc_now())
    save_json(path,obj); return obj


def capture_all_formats():
    rows=[]; manifest={}
    for codebook in FORMAT_SYMBOLS:
        for family in tqdm(FAMILIES,desc=f'{codebook}: source and untreated inputs'):
            obj=capture_format_family(family,codebook); rows.extend(obj['rows'])
            name=family['family_id']+'__'+codebook+'.json'
            manifest[name]=file_hash(PHASE_ROOT/'pre_intervention'/name)
    assert len(rows)==3456 and len({r['trial_id'] for r in rows})==3456
    freeze(PHASE_ROOT/'pre_intervention_manifest.json',manifest)
    return rows


def freeze_format_predictions(rows):
    identity=dict(runtime_sha256=file_hash(PHASE_ROOT/'runtime_identity.json'),
        frozen_models_sha256=file_hash(MODEL_ROOT/'frozen_predictors.json'),
        pre_manifest_sha256=file_hash(PHASE_ROOT/'pre_intervention_manifest.json'),
        protocol_sha256=file_hash(RUN_ROOT/'protocol.json'),source_identity_sha256=file_hash(MODEL_ROOT/'source_identity.json'))
    sha=seal_predictions(PHASE_ROOT/'predictions_before_intervention.json',rows,FITTED,identity)
    freeze(PHASE_ROOT/'prediction_seal.json',dict(identity,prediction_sha256=sha))
    print('All ABC/XYZ/DIGITS forecasts sealed:',sha)


def verify_format_seal():
    seal=read_json(PHASE_ROOT/'prediction_seal.json')
    for key,path in [('prediction_sha256',PHASE_ROOT/'predictions_before_intervention.json'),
                     ('runtime_sha256',PHASE_ROOT/'runtime_identity.json'),
                     ('frozen_models_sha256',MODEL_ROOT/'frozen_predictors.json'),
                     ('pre_manifest_sha256',PHASE_ROOT/'pre_intervention_manifest.json'),
                     ('protocol_sha256',RUN_ROOT/'protocol.json'),
                     ('source_identity_sha256',MODEL_ROOT/'source_identity.json')]:
        assert file_hash(path)==seal[key], 'Sealed input changed: '+str(path)
    return seal


def execute_format_targets():
    seal=verify_format_seal(); sealed_hash=digest(seal)
    manifest=read_json(PHASE_ROOT/'pre_intervention_manifest.json')
    folder=PHASE_ROOT/'interventions'; folder.mkdir(parents=True,exist_ok=True)
    all_rows=[]; tail=[]
    for cb in FORMAT_SYMBOLS:
        for family in tqdm(FAMILIES,desc=f'{cb}: fixed interventions'):
            stem=family['family_id']+'__'+cb
            path=PHASE_ROOT/'pre_intervention'/(stem+'.json'); obj=read_json(path)
            assert file_hash(path)==manifest[path.name] and file_hash(path.with_suffix('.npz'))==obj['arrays_sha256']
            with np.load(path.with_suffix('.npz'),allow_pickle=False) as arrays:
                for row in obj['rows']:
                    dest=folder/(row['trial_id']+'.json')
                    identity=digest([RUNTIME_IDENTITY,row,sealed_hash,obj['arrays_sha256']])
                    if dest.exists():
                        saved=read_json(dest); assert saved['identity']==identity
                    else:
                        ex=make_format_example(family,row['recipient'],'outcome',row['recipient_mapping'],cb)
                        baseline=obj['baseline_records'][row['recipient_key']]
                        payload=arrays[row['payload_key']]; pre=arrays[row['pre_key']]
                        assert state_hash(payload)==row['payload_sha256'] and state_hash(pre)==row['pre_sha256']
                        # Source-free terminal replay is computed before the real patch, but is
                        # a deterministic output-path reconstruction, not a competing fitted predictor.
                        head_scores=TAIL.scores(bf16(pre+payload),cb).tolist() if row['layer']==EXPECTED_LAYER_COUNT-1 else None
                        record,_=predictive_forward(ex,layer=row['layer'],payload=payload,
                            expected_pre=pre,expected_input=baseline['input_sha256'])
                        saved=dict(identity=identity,record=record,tail_scores=head_scores,
                            sealed_prediction_identity=sealed_hash,created_at_utc=utc_now())
                        save_json(dest,saved)
                    actual=center(saved['record']['candidate_logprobs'])-row['s0']
                    all_rows.append(dict(row,actual_delta=actual.tolist(),post_record=saved['record']))
                    if saved['tail_scores'] is not None:
                        tail.append(dict(trial_id=row['trial_id'],family_id=row['family_id'],codebook=cb,
                            max_abs_error=float(np.max(np.abs(np.asarray(saved['tail_scores'])-center(saved['record']['candidate_logprobs'])))),
                            post_answer_match=int(np.argmax(saved['tail_scores'])==np.argmax(saved['record']['candidate_logprobs']))))
    assert len(all_rows)==3456 and len({r['trial_id'] for r in all_rows})==3456
    save_json(PHASE_ROOT/'target_rows.json',all_rows)
    pd.DataFrame(tail).to_csv(PHASE_ROOT/'analysis/current_tail_replay.csv',index=False)
    return all_rows


def compare_current_abc_to_archive(rows):
    old={r['trial_id']:r for r in PRIOR.old_rows}; reports=[]
    for r in rows:
        if r['codebook']!='ABC': continue
        prior=old[r['base_trial_id']]
        reports.append(dict(trial_id=r['base_trial_id'],family_id=r['family_id'],layer=r['layer'],
            baseline_max_abs_error=float(np.max(np.abs(np.asarray(r['s0'])-prior['s0']))),
            effect_max_abs_error=float(np.max(np.abs(np.asarray(r['actual_delta'])-prior['actual_delta']))),
            before_state_identical=r['pre_sha256']==prior['pre_sha256'],source_delta_identical=r['payload_sha256']==prior['payload_sha256']))
    pd.DataFrame(reports).to_csv(PHASE_ROOT/'analysis/archived_vs_current_ABC.csv',index=False)
    print('Cross-format comparisons use the fresh ABC anchor. Archive/runtime differences are reported separately.')


def export_transfer():
    marker=PHASE_ROOT/'completion.json'
    status=read_json(marker)['status'] if marker.exists() else 'partial_not_complete'
    # Include the old evidence subset actually used, without duplicating unrelated physical trials.
    snapshot=MODEL_ROOT/'source_snapshot'; snapshot.mkdir(parents=True,exist_ok=True)
    for p in PRIOR_CACHE.glob('*'):
        if p.is_file() and p.suffix in ('.json','.npz'):
            dest=snapshot/p.name
            if not dest.exists() or file_hash(dest)!=file_hash(p): shutil.copy2(p,dest)
    selected=[RUN_ROOT/'protocol.json',RUN_ROOT/'support.json']
    selected += [p for p in MODEL_ROOT.rglob('*') if p.is_file() and '.tmp' not in p.name and p.suffix!='.zip']
    manifest={str(p.relative_to(RUN_ROOT)):file_hash(p) for p in sorted(set(selected))}
    archive=RUN_ROOT/f'VisualTorque_Frozen_Format_Transfer_{MODEL_KEY}_Output.zip'
    temp=archive.with_suffix('.tmp.zip')
    with zipfile.ZipFile(temp,'w',zipfile.ZIP_DEFLATED) as z:
        for name in manifest: z.write(RUN_ROOT/name,name)
        z.writestr('export_manifest.json',json.dumps(dict(status=status,files=manifest,created_at_utc=utc_now()),indent=2))
    temp.replace(archive)
    print('Saved:',archive,'|',status)
    try:
        from google.colab import files
        files.download(str(archive))
    except Exception:
        print('Browser download unavailable. The ZIP remains at the Drive path above.')
    return archive


In [ ]:
%%cell_progress 7/17 Render and verify the reused images
prepare_transfer_images()
images={r['image_filename'] for f in FAMILIES for r in f['roles'].values()}
for name in tqdm(sorted(images),desc='Verify unchanged image bytes'):
    assert file_hash(IMAGE_DIR/name)==PRIOR.image_manifest[name], 'Renderer/image mismatch: '+name
freeze(RUN_ROOT/'image_manifest.json',{name:PRIOR.image_manifest[name] for name in sorted(images)})
budget=[]
for cb in FORMAT_SYMBOLS:
    clean=0; target=0
    for family in FAMILIES:
        ids=set()
        for spec in format_trial_specs(family,cb):
            target+=1
            for role,task,mapping in [(spec['source_before'],spec['task'],spec['source_mapping']),
                                      (spec['source_after'],spec['task'],spec['source_mapping']),
                                      (spec['recipient'],'outcome',spec['recipient_mapping'])]:
                ids.add((role,task,digest(mapping)))
        clean+=len(ids)
    budget.append(dict(codebook=cb,input_forwards=clean,intervention_forwards=target,total=clean+target))
display(pd.DataFrame(budget))
print('Tail replays use only the output modules and are additional to these full-model forwards.')


In [ ]:
%%cell_progress 8/17 Check GPU and checkpoint access
cell_stage('Checking CUDA device')
assert torch.cuda.is_available(), 'Select a GPU runtime with sufficient BF16 memory.'
print(torch.cuda.get_device_name(0))
print('CUDA memory (GB):', round(
    torch.cuda.get_device_properties(0).total_memory / 2**30, 1
))

cell_stage('Authenticating with Hugging Face')
from google.colab import userdata
from huggingface_hub import login
try:
    token = userdata.get('HF_TOKEN')
    if token:
        login(token=token, add_to_git_credential=False)
        print('Hugging Face login complete.')
    else:
        print('HF_TOKEN is not set. Public checkpoints may still work.')
except Exception as exc:
    print('Hugging Face login unavailable:', type(exc).__name__)


In [ ]:
%%cell_progress 9/17 Load pinned model
cell_stage(f'Loading {MODEL_KEY} weights for frozen format transfer')
import gc
if 'adapter' in globals():
    del adapter
    gc.collect()
    torch.cuda.empty_cache()

adapter = HuggingFaceMultimodalAdapter(
    MODEL_ID, revision=MODEL_REVISION,
    dtype=MODEL_DTYPE, device_map=DEVICE_MAP,
)
display(adapter.metadata)
assert adapter.metadata['resolved_revision'] == MODEL_REVISION

def resolve_module_path(root, path):
    value = root
    for component in path.split('.'):
        value = getattr(value, component)
    return value

def discover_decoder_layers(model):
    candidates = (
        'model.language_model.layers', 'model.text_model.layers',
        'language_model.model.layers', 'model.model.layers',
        'model.layers', 'language_model.layers',
    )
    for path in candidates:
        try:
            layers = resolve_module_path(model, path)
        except AttributeError:
            continue
        if isinstance(layers, torch.nn.ModuleList) and len(layers) >= 8:
            return path, layers
    raise RuntimeError('Could not locate decoder layers.')

LAYER_PATH, DECODER_LAYERS = discover_decoder_layers(adapter.model)
assert len(DECODER_LAYERS) == EXPECTED_LAYER_COUNT
assert all(0 <= layer < len(DECODER_LAYERS) for layer in CAPTURE_LAYERS)
print('Layer path:', LAYER_PATH)
print('Capture layers:', CAPTURE_LAYERS)


In [ ]:
%%cell_progress 10/17 Freeze runtime and candidate scoring
adapter.model.eval()
assert not any(str(v) in ('disk','cpu') for v in getattr(adapter.model,'hf_device_map',{}).values()), 'Full BF16 model must fit in GPU memory.'
torch.manual_seed(SEED); np.random.seed(SEED); torch.backends.cuda.matmul.allow_tf32=False
RUNTIME_IDENTITY=current_runtime_identity()
assert RUNTIME_IDENTITY['transformers']==PRIOR.runtime['transformers'], 'Use the archived Transformers version.'
critical=('behavior/adapter.py','behavior/data.py','render.py','world.py')
for suffix in critical:
    matches=[k for k in PRIOR.runtime['toolkit'] if k.endswith('/'+suffix)]
    assert matches, 'Missing old toolkit identity: '+suffix
    for key in matches:
        assert RUNTIME_IDENTITY['toolkit'].get(key)==PRIOR.runtime['toolkit'][key], 'Critical toolkit source changed: '+key
RUNTIME_IDENTITY.update(model_id=MODEL_ID,model_revision=MODEL_REVISION,gpu=torch.cuda.get_device_name(0),
    numpy=np.__version__,pandas=pd.__version__,adapter_metadata=adapter.metadata,
    protocol_sha256=file_hash(RUN_ROOT/'protocol.json'),source_identity_sha256=file_hash(MODEL_ROOT/'source_identity.json'))
freeze(PHASE_ROOT/'runtime_identity.json',RUNTIME_IDENTITY)
probe=np.array([0.,1.,-1.,1.00390625,1.01171875,50.21,-81.37],dtype=np.float32)
assert np.array_equal(bf16(probe),torch.tensor(probe).bfloat16().float().numpy())
TAIL=TailReadout()
print('Candidate token plans:'); display(pd.DataFrame(TAIL.plans).T)
print('Current runtime frozen. All new-format inputs and interventions use this same runtime.')


In [ ]:
%%cell_progress 11/17 Audit archived final-layer output path
run_archive_tail_audit()


In [ ]:
%%cell_progress 12/17 Capture all three formats before intervention
PRE_ROWS=capture_all_formats()


In [ ]:
%%cell_progress 13/17 Seal all frozen predictions
freeze_format_predictions(PRE_ROWS)


In [ ]:
%%cell_progress 14/17 Execute fixed interventions
TARGET_ROWS=execute_format_targets()


In [ ]:
%%cell_progress 15/17 Analyze format transfer and prediction failures
verify_format_seal()
sealed=read_json(PHASE_ROOT/'predictions_before_intervention.json')
summary,tests=analyze_transfer(TARGET_ROWS,sealed,FITTED,ANALYSIS_DIR,MODEL_KEY)
compare_current_abc_to_archive(TARGET_ROWS)
display(summary[summary.frozen_selected].round(4)); display(tests.round(4))
clean=[]
for path in (PHASE_ROOT/'pre_intervention').glob('*.json'):
    obj=read_json(path)
    clean.extend(dict(r,family_id=obj['family_id'],codebook=obj['codebook']) for r in obj['tail_clean'])
pd.DataFrame(clean).to_csv(ANALYSIS_DIR/'current_clean_tail_replay.csv',index=False)
save_json(PHASE_ROOT/'completion.json',dict(status='frozen_format_transfer_complete',n_families=96,
    n_codebooks=3,n_trials=len(TARGET_ROWS),prediction_sha256=file_hash(PHASE_ROOT/'predictions_before_intervention.json'),
    frozen_predictors_sha256=file_hash(MODEL_ROOT/'frozen_predictors.json'),
    interpretation='new formats on reused families; not new geometry, no equivalence claim'))
print('Complete. Accuracy here predicts the model choice; physical correctness is reported separately.')


In [ ]:
%%cell_progress 16/17 Combine planned model comparisons
frames=[]
for key in MODEL_SPECS:
    folder=RUN_ROOT/'models'/key/'validation'
    if (folder/'completion.json').exists():
        df=pd.read_csv(folder/'analysis/primary_comparisons.csv',keep_default_na=False,na_values=[''])
        assert len(df)==16; frames.append(df)
if len(frames)==2:
    both=pd.concat(frames,ignore_index=True)
    both['holm_p_across_models']=holm(both.p_value)
    both.to_csv(ANALYSIS_DIR/'cross_model_primary_comparisons.csv',index=False)
    display(both.round(4))
else:
    print('Cross-model correction pending: run the other model in a fresh runtime using the same RUN_ID.')


In [ ]:
%%cell_progress 17/17 Export complete or partial results
OUTPUT_ZIP=export_transfer()
